# 🎬 CloneCinema — ComfyUI + WAN 2.2 TI2V 5B (Kaggle T4 x2)

**Modo Warm Session con Auto-Apagado Inteligente (Watchdog)**

- **Dataset Instantáneo**: Los pesos (~17 GB) se montan en 0 segundos desde `/kaggle/input/wan-2-2-5b-model-data`.
- **Autorregistro en Supabase**: Publica automáticamente la URL del túnel Cloudflare en la base de datos de CloneCinema.
- **Protección de 30 Horas (Watchdog)**: Si pasan 15 minutos sin generar video, el servidor se auto-apaga para no gastar cuota de GPU.
- **Bucket Supabase**: Soporta recepción de imágenes vía URL de Supabase (`wan_inputs` o `keyframes`).

In [ ]:
# ⚡ [1/5] Clonar ComfyUI e instalar dependencias
import os, sys, subprocess

print("⚡ [1/5] Instalando dependencias base y ComfyUI en /kaggle/working...")
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "aria2", "-qq"], stdout=subprocess.DEVNULL)

os.chdir("/kaggle/working")
if not os.path.exists("/kaggle/working/ComfyUI"):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/kaggle/working/ComfyUI"], check=True)

os.chdir("/kaggle/working/ComfyUI")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)

# Instalar nodos esenciales
os.chdir("/kaggle/working/ComfyUI/custom_nodes")
nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git")
]
for name, repo in nodes:
    if not os.path.exists(name):
        subprocess.run(["git", "clone", repo, name, "-q"], check=True)

os.chdir("/kaggle/working/ComfyUI")
print("✅ ComfyUI y dependencias listos en /kaggle/working/ComfyUI.")

In [ ]:
# 📥 [2/5] Carga ultrarrápida de modelos Wan 2.2 desde Dataset Kaggle
import os, subprocess, shutil

os.chdir("/kaggle/working/ComfyUI")
os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("input", exist_ok=True)

dataset_path = "/kaggle/input/wan-2-2-5b-model-data"

if os.path.exists(dataset_path):
    print("⚡ ¡Dataset Kaggle detectado! Enlazando pesos instantáneamente (0 segundos)...")
    # 1. Diffusion Model (wan2.2_ti2v_5B_fp16.safetensors)
    src_diff = os.path.join(dataset_path, "wan2.2_ti2v_5B_fp16.safetensors")
    dst_diff = "models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"
    if os.path.exists(src_diff) and not os.path.exists(dst_diff):
        os.symlink(src_diff, dst_diff)
        print("   🔗 Diffusion Model enlazado.")

    # 2. VAE (wan2.2_vae.safetensors)
    src_vae = os.path.join(dataset_path, "wan2.2_vae.safetensors")
    dst_vae = "models/vae/wan2.2_vae.safetensors"
    if os.path.exists(src_vae) and not os.path.exists(dst_vae):
        os.symlink(src_vae, dst_vae)
        print("   🔗 VAE enlazado.")

    # 3. Text Encoder (umt5_xxl_fp8_e4m3fn_scaled.safetensors)
    src_clip = os.path.join(dataset_path, "umt5_xxl_fp8_e4m3fn_scaled.safetensors")
    dst_clip = "models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
    if os.path.exists(src_clip) and not os.path.exists(dst_clip):
        os.symlink(src_clip, dst_clip)
        print("   🔗 Text Encoder T5 enlazado.")
    print("✅ Todos los modelos Wan 2.2 enlazados en 0.05 segundos.")
else:
    print("⚠️ Dataset no encontrado en /kaggle/input, descargando vía aria2c multihilo...")
    downloads = [
        ("models/diffusion_models", "wan2.2_ti2v_5B_fp16.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"),
        ("models/vae", "wan2.2_vae.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors"),
        ("models/text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors")
    ]
    for out_dir, filename, url in downloads:
        dest = os.path.join(out_dir, filename)
        if not os.path.exists(dest):
            print(f"🚀 Descargando {filename}...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--file-allocation=none", "-d", out_dir, "-o", filename, url], check=True)


In [ ]:
# 🌐 [3/5] Instalar binario de Cloudflare Tunnel
import os, subprocess
os.chdir("/kaggle/working/ComfyUI")
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅ Cloudflared listo en /kaggle/working/ComfyUI.")

In [ ]:
# 🚀 [4/5] Levantar servidor ComfyUI en GPU
import subprocess, time, os, sys
os.chdir("/kaggle/working/ComfyUI")
print("🚀 Levantando servidor ComfyUI en GPU T4...")
subprocess.Popen([sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188", "--highvram", "--fast"])
time.sleep(12)
subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "ComfyUI HTTP Status: %{http_code}\n", "http://localhost:8188"])

In [ ]:
# 🛡️ [5/5] Túnel Cloudflare + Autorregistro en Supabase + Watchdog Auto-Apagado
import subprocess, time, re, json, urllib.request, threading, os, sys, base64

SUPABASE_URL = "https://micjwzatshemmovaneix.supabase.co"
try:
    from kaggle_secrets import UserSecretsClient
    SUPABASE_KEY = UserSecretsClient().get_secret("SUPABASE_SERVICE_ROLE_KEY")
except Exception:
    SUPABASE_KEY = os.environ.get("SUPABASE_KEY") or base64.b64decode("c2Jfc2VjcmV0XzFPOXV4QnoyM080cEhUYWxXVkdObHdfYU5HT09MQjU=").decode("utf-8")
TUNNEL_NODE_ID = "a0000000-0000-0000-0000-000000000001"
IDLE_TIMEOUT_MINUTES = 20  # Apagado tras 20 min sin peticiones

def register_supabase(tunnel_url, status="running", note=""):
    try:
        payload = {
            "id": TUNNEL_NODE_ID,
            "kind": "video",
            "status": status,
            "provider": "wan22_tunnel",
            "input_url": tunnel_url,
            "prompt": f"Kaggle Wan 2.2 Live Gateway - {note}",
            "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        req = urllib.request.Request(
            f"{SUPABASE_URL}/rest/v1/ai_jobs?on_conflict=id",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "apikey": SUPABASE_KEY,
                "Authorization": f"Bearer {SUPABASE_KEY}",
                "Content-Type": "application/json",
                "Prefer": "resolution=merge-duplicates"
            },
            method="POST"
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            print(f"📡 [Supabase] Estado sincronizado: status={status} (HTTP {resp.status})")
    except Exception as e:
        print(f"⚠️ Error sincronizando con Supabase: {e}")

subprocess.run(["pkill", "-f", "cloudflared"])
proc = subprocess.Popen(
    ["/kaggle/working/ComfyUI/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
url = None
for _ in range(120):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

if not url:
    print("❌ No se pudo obtener la URL de Cloudflare.")
    sys.exit(1)

print("\n" + "="*70)
print("🎉 ¡COMFYUI WAN 2.2 TI2V 5B ACTIVO!")
print(f"👉 ENLACE PÚBLICO: {url}")
print("="*70 + "\n")

# Registrar URL en Supabase
register_supabase(url, status="running", note="Activo y listo para generar")

# Watchdog de Inactividad para proteger las 30 horas de GPU
def watchdog_loop():
    last_job_count = 0
    last_activity = time.time()
    print(f"🛡️ Watchdog activado: Auto-apagado tras {IDLE_TIMEOUT_MINUTES} min de inactividad.")
    while True:
        time.sleep(60)
        try:
            req = urllib.request.Request("http://127.0.0.1:8188/history")
            with urllib.request.urlopen(req, timeout=5) as r:
                history = json.loads(r.read().decode("utf-8"))
                count = len(history)
                if count != last_job_count:
                    last_job_count = count
                    last_activity = time.time()
                    print(f"⚡ Actividad detectada: {count} trabajos en historial.")
        except Exception:
            pass
        
        idle_sec = time.time() - last_activity
        if idle_sec > IDLE_TIMEOUT_MINUTES * 60:
            print(f"🛑 INACTIVIDAD: Han pasado {IDLE_TIMEOUT_MINUTES} minutos sin peticiones.")
            print("💤 Apagando servidor Kaggle para conservar tus 30 horas semanales de GPU...")
            register_supabase(url, status="done", note="Apagado automático por inactividad")
            os._exit(0)

threading.Thread(target=watchdog_loop, daemon=True).start()

try:
    while True:
        time.sleep(15)
except KeyboardInterrupt:
    print("🛑 Servidor detenido manualmente.")
    register_supabase(url, status="done", note="Detenido manualmente por usuario")
